# 03 · Unregularized Inversion: Least Squares and Why It Fails

The forward model (notebook 01) predicts data from slip: $\mathbf{d} = G\mathbf{m}$.
*Inversion* runs it backwards — estimate the slip $\mathbf{m}$ from noisy data
$\mathbf{d}$. This notebook does that the most direct way, by **least squares**,
and shows why raw least squares alone is not enough.

## Learning objectives
- Set up the linear inverse problem $\mathbf{d} = G\mathbf{m} + \boldsymbol{\varepsilon}$.
- Solve it by (weighted) least squares with `geodef.invert()`.
- Read the fit with `InversionResult` and `plot.fit`.
- See an **unregularized** inversion overfit the noise and oscillate wildly —
  the motivation for regularization in notebook 04.

## The linear inverse problem

Data are the forward model plus noise:

$$ \mathbf{d} = G\,\mathbf{m} + \boldsymbol{\varepsilon}, \qquad
\operatorname{cov}(\boldsymbol{\varepsilon}) = C_d. $$

**Ordinary least squares (OLS)** picks the slip that minimizes the squared
misfit $\lVert G\mathbf{m} - \mathbf{d}\rVert^2$. Setting the gradient to zero
gives the *normal equations*:

$$ G^{\mathsf T} G\, \hat{\mathbf{m}} = G^{\mathsf T}\mathbf{d}. $$

**Weighted least squares (WLS)** accounts for the fact that some data are more
precise than others. With the weight matrix $W = C_d^{-1}$,

$$ \hat{\mathbf{m}} = \left(G^{\mathsf T} W G\right)^{-1} G^{\mathsf T} W \mathbf{d}. $$

Down-weighting noisy observations keeps them from dominating the fit. `geodef`
builds $W$ from each dataset's uncertainties automatically, so `invert()`
defaults to WLS.

The catch is the matrix $G^{\mathsf T} W G$. If it is **ill-conditioned** —
nearly singular — then small changes in the data (i.e. noise) cause large
changes in $\hat{\mathbf{m}}$. Deep and adjacent patches produce almost the
same surface signal, so their columns of $G$ are nearly parallel and the
inverse amplifies noise. We will see exactly that below.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import geodef

%load_ext autoreload
%autoreload 2

rng = np.random.default_rng(0)

# --- Recurring teaching scenario -------------------------------------------
# The higher-resolution megathrust from notebook 01, reused (copied) across
# notebooks 03-10 so every inversion targets the same "true" slip model.
fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=25e3,   # centroid 25 km deep
    strike=315.0, dip=15.0,            # NW-striking, shallow dip
    length=180e3, width=90e3,          # 180 km x 90 km
    n_length=12, n_width=6,            # -> 72 patches
)
N = fault.n_patches
nL, nW = fault.grid_shape

# "True" slip: notebook 01's smooth Gaussian thrust asperity, dip-slip only.
# The model vector is blocked as [strike-slip (N) | dip-slip (N)], so the
# strike-slip half is zero.
i = np.arange(N) % nL
j = np.arange(N) // nL
i0, j0 = (nL - 1) / 2, nW / 2
bump = np.exp(-(((i - i0) / 3.0) ** 2 + ((j - j0) / 1.5) ** 2))
slip_true = np.concatenate([np.zeros(N), 3.0 * bump])

# A grid of surface GNSS stations (note: GNSS takes lon, lat order).
glon, glat = np.meshgrid(np.linspace(98.5, 101.5, 8), np.linspace(-3.6, -0.4, 8))
glon, glat = glon.ravel(), glat.ravel()
n_sta = glon.size

# Synthetic data: forward-model the true slip, then add seeded Gaussian noise.
_zero = np.zeros(n_sta)
_one = np.ones(n_sta)
G_full = geodef.greens.greens(
    fault, geodef.GNSS(glon, glat, _zero, _zero, _zero, _one, _one, _one)
)
sigma = 0.01  # 1 cm station noise
d_obs = G_full @ slip_true + rng.normal(0.0, sigma, G_full.shape[0])
gnss = geodef.GNSS(
    glon, glat,
    ve=d_obs[0::3], vn=d_obs[1::3], vu=d_obs[2::3],
    se=np.full(n_sta, sigma), sn=np.full(n_sta, sigma), su=np.full(n_sta, sigma),
)
print(f"{N} patches, {n_sta} stations, {d_obs.size} observations")

40 patches, 36 stations, 108 observations


## The data we are trying to explain

Here is the true slip we are pretending not to know, and the noisy GNSS
displacements it produced. Our job: recover the left panel from the arrows on
the right.

In [2]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
geodef.plot.slip(fault, slip_true[N:], ax=ax1, title="True dip-slip (unknown)",
                 coords="geographic",
                 colorbar_label="Slip (m)")
geodef.plot.vectors(gnss, fault, ax=ax2, title="Observed GNSS (with noise)")
plt.tight_layout()

## Solve it: weighted least squares

Because the true slip is pure dip-slip, we solve for one slip component per
patch with `components='dip'` (notebook 08 revisits component choices). With no
`smoothing` argument, `invert()` performs a plain weighted least-squares solve
— no regularization, no prior.

In [3]:
raw = geodef.invert(fault, gnss, components="dip")
print(f"method:  weighted least squares (unregularized)")
print(f"reduced chi^2: {raw.chi2:.3f}")
print(f"RMS residual:  {raw.rms * 1000:.2f} mm")

method:  weighted least squares (unregularized)
reduced chi^2: 0.802
RMS residual:  7.11 mm


A reduced $\chi^2$ near (or below) 1 means the model fits the data to within
their uncertainties. That sounds like success — until we look at the slip.

In [4]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
lim_true = slip_true[N:].max()
lim_raw = abs(raw.slip_vector).max()
geodef.plot.slip(fault, slip_true[N:], ax=ax1, cmap="RdBu_r",
                 vmin=-lim_true, vmax=lim_true, title="True slip",
                 colorbar_label="Slip (m)")
geodef.plot.slip(fault, raw.slip_vector, ax=ax2, cmap="RdBu_r",
                 vmin=-lim_raw, vmax=lim_raw, title="Unregularized WLS",
                 colorbar_label="Slip (m)")
plt.tight_layout()

## Overfitting: a "perfect" fit to nonsense

The recovered slip is wild — swinging positive and negative by far more than the
true peak, with a checkerboard pattern that has no physical meaning. Yet it fits
the data slightly *better* than the truth would. It is fitting the **noise**.

Compare the numbers:

In [5]:
def roughness(m):
    return float(np.std(np.diff(m)))

print(f"true slip:    peak {slip_true[N:].max():5.2f} m,  roughness {roughness(slip_true[N:]):.3f}")
print(f"recovered:    peak {raw.slip_vector.max():5.2f} m,  roughness {roughness(raw.slip_vector):.3f}")
print()
G_dip = G_full[:, N:]
print(f"condition number of G^T G : {np.linalg.cond(G_dip.T @ G_dip):.1e}")

true slip:    peak  1.88 m,  roughness 0.301
recovered:    peak 20.31 m,  roughness 12.472

condition number of G^T G : 1.0e+06


The condition number is enormous, so the inverse amplifies the 1 cm of station
noise into metres of spurious, oscillating slip. The fit to the *data* is fine
(`plot.fit` below) — the model is simply not trustworthy.

In [6]:
geodef.plot.fit(gnss.obs, raw.predicted,
                title="Observed vs. predicted (fits the noise well)")

<Axes: title={'center': 'Observed vs. predicted (fits the noise well)'}, xlabel='Observed', ylabel='Predicted'>

## The lesson

Fitting the data as well as possible is **not** the goal — the data contain
noise we do *not* want to reproduce. An ill-posed inversion will happily fit
that noise with a physically absurd model. The cure is to add prior information
that favors smooth, sensible slip: **regularization**, the subject of
notebook 04.

## Exercises
1. **More noise.** Change `sigma` in the setup to `0.03` and re-run. Does the
   unregularized slip get better or worse?
2. **Fewer stations.** Replace the 6×6 station grid with a 4×4 grid. What
   happens to the condition number and the recovered slip?
3. **Both components.** Solve with `components='both'` instead of `'dip'`. The
   problem now has $2N$ unknowns — is it more or less stable?

## Checkpoint questions
1. Why can a model have a great $\chi^2$ but still be useless?
2. What property of $G^{\mathsf T} W G$ controls how badly noise is amplified?
3. Why does weighting by $C_d^{-1}$ matter when stations have different
   uncertainties?

## Summary
- Inversion estimates slip from data by minimizing weighted misfit.
- WLS has a closed form, $\hat{\mathbf{m}} = (G^{\mathsf T} W G)^{-1} G^{\mathsf T} W \mathbf{d}$.
- When $G^{\mathsf T} W G$ is ill-conditioned, the solution overfits noise.

**Next:** notebook 04 stabilizes the inversion with regularization.